Практична робота 8
Групування і зведені таблиці: groupby, agg, pivot_table, crosstab
Виконавець: Войтович Богдан, IT-32
Варіант: 6 (Чернігів)

In [56]:
import numpy as np
import pandas as pd

np.random.seed(42)               # відтворюваність

base_temp = 8.0                 # середньорічна температура для Чернігова
amplitude = 13.0                # сезонна амплітуда
city = "Чернігів"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)   # пік у липні (7 місяць)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        rows.append({"місто": city, "рік": year, "місяць": month, "температура": temp})

climate = pd.DataFrame(rows)


In [57]:
year_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
print("Завдання 1 - статистика за роками:\n", year_stats)


Завдання 1 - статистика за роками:
           mean  min   max
рік                      
2021  8.283333 -4.5  22.6
2022  7.408333 -5.2  20.1
2023  7.808333 -5.5  21.1
2024  7.666667 -5.2  20.9


In [58]:
month_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
print("\nЗавдання 2 - статистика за місяцями:\n", month_stats)

max_std_month = month_stats["std"].idxmax()
print(f"\nМісяць із найбільшим розкидом (std): {max_std_month}")



Завдання 2 - статистика за місяцями:
           mean       std
місяць                  
1       -4.900  0.424264
2       -4.225  1.132475
3        0.600  1.023067
4        8.375  0.865544
5       14.225  0.727438
6       19.250  0.300000
7       21.000  1.116542
8       19.475  1.408013
9       14.375  1.250000
10       7.625  0.689807
11       1.475  0.618466
12      -3.775  1.135415

Місяць із найбільшим розкидом (std): 8


In [59]:
pivot_tbl = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
print("\nЗавдання 3 - зведена таблиця:\n", pivot_tbl)



Завдання 3 - зведена таблиця:
 рік     2021  2022  2023  2024
місяць                        
1       -4.5  -4.8  -5.5  -4.8
2       -3.4  -5.2  -3.1  -5.2
3        2.1  -0.2   0.3   0.2
4        9.5   7.4   8.4   8.2
5       14.3  13.5  13.9  15.2
6       19.0  19.6  19.0  19.4
7       22.6  20.1  20.4  20.9
8       20.0  17.8  21.1  19.0
9       14.0  16.0  14.5  13.0
10       8.5   7.8   6.9   7.3
11       1.0   1.6   2.3   1.0
12      -3.7  -4.7  -4.5  -2.2


In [60]:
# Додавання сезону
def season(month):
    if month in [12, 1, 2]:
        return "Зима"
    elif month in [3, 4, 5]:
        return "Весна"
    elif month in [6, 7, 8]:
        return "Літо"
    else:
        return "Осінь"

climate["сезон"] = climate["місяць"].apply(season)

# Додати булеву категорію "тепліше_за_середнє"
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"])
print("\nЗавдання 4 - crosstab сезону і теплоти:\n", ct)



Завдання 4 - crosstab сезону і теплоти:
 тепліше_за_середнє  False  True 
сезон                           
Весна                   5      7
Зима                   12      0
Літо                    0     12
Осінь                   7      5


In [61]:
try:
    pivot_result = climate.pivot(index="місяць", columns="рік", values="температура")
    print("\nЗавдання 5 - pivot() результат:\n", pivot_result)
except Exception as e:
    print("pivot() помилка:", e)



Завдання 5 - pivot() результат:
 рік     2021  2022  2023  2024
місяць                        
1       -4.5  -4.8  -5.5  -4.8
2       -3.4  -5.2  -3.1  -5.2
3        2.1  -0.2   0.3   0.2
4        9.5   7.4   8.4   8.2
5       14.3  13.5  13.9  15.2
6       19.0  19.6  19.0  19.4
7       22.6  20.1  20.4  20.9
8       20.0  17.8  21.1  19.0
9       14.0  16.0  14.5  13.0
10       8.5   7.8   6.9   7.3
11       1.0   1.6   2.3   1.0
12      -3.7  -4.7  -4.5  -2.2


Контрольні питання (короткі відповіді)
Split-apply-combine:
Групування розбиває дані на підгрупи (split), до кожної застосовує функцію агрегації (apply), потім збирає результат у єдину таблицю (combine).

pivot() vs pivot_table():
pivot() не дозволяє дублікати в індексах, тому дає помилку, якщо для однієї пари (index, columns) є кілька рядків. pivot_table() агрегує значення (через aggfunc), тому працює навіть з дублями.

crosstab() vs groupby(...).size():
crosstab() підраховує частоти за двома (або більше) категоріями у вигляді зведеної таблиці, це зручний метод для обчислення перехресних частот. groupby.size() повертає підрахунок для груп, але його потрібно додатково трансформувати у матричний вигляд.

Чому long (tidy) формат є обов’язковим:
Тільки в довгому форматі кожне спостереження — один рядок, що дозволяє коректно працювати з groupby, pivot_table і іншими функціями. Широкий формат із місяцями чи роками у заголовках не підходить для програмної агрегації.